# 03. 분석 결과를 웹 서비스로 — Streamlit 대시보드 만들기와 GitHub 배포

01 · 02 노트북의 분석은 노트북을 연 사람만 볼 수 있습니다. 이번에는 분석 결과를 **누구나 인터넷 주소(URL) 하나로 보는 대시보드**로 만들어 배포합니다.

```
 [원본 CSV 124MB] ─(1) 가공─▶ [작은 데이터 · 약 2.4MB] ─(2)─▶ [app.py 대시보드] ─(3) 로컬 테스트
                                                                     │
                       (6) 인터넷 주소로 공개 ◀─(5) Streamlit Cloud ◀─(4) GitHub 에 올리기(git push)
```

<div style="border:1px solid #9ABBE0;border-left:5px solid #1F3A7A;background:#EAF2FD;padding:10px 14px;border-radius:4px">

**이 노트북의 학습 목표**

1. 배포에 맞게 데이터를 **작고 가볍게** 가공해 저장한다.
2. **Streamlit** 으로 필터 · 지표 · 탭 · 그래프가 있는 대시보드(`app.py`)를 만든다.
3. 앱을 **자동 테스트(AppTest)** 와 로컬 실행으로 확인한다.
4. **git** 으로 버전을 기록하고 **GitHub** 에 올린 뒤, **Streamlit Community Cloud** 로 무료 배포한다.
5. 코드를 고친 뒤 다시 push 하면 **자동으로 재배포**되는 흐름을 이해한다.

</div>

## 0. 준비물

| 준비물 | 확인 방법 | 없으면 |
|---|---|---|
| Python 패키지 `streamlit` · `pandas` · `plotly` | 아래 셀 실행 | 터미널에서 `pip install streamlit pandas plotly` |
| **git** | 아래 셀 실행 (`git version …` 출력) | https://git-scm.com 에서 설치 후 VS Code · 터미널 재시작 |
| **GitHub 계정** | https://github.com 로그인 | 무료 가입 |
| Streamlit Community Cloud | https://share.streamlit.io 에 GitHub 계정으로 로그인 | 무료 (GitHub 계정으로 가입) |

In [1]:
import importlib, subprocess

for name in ["pandas", "plotly", "streamlit"]:
    try:
        module = importlib.import_module(name)
        print(f"✓ {name:10s} {module.__version__}")
    except ImportError:
        print(f"✗ {name:10s} 설치 필요 →  pip install {name}")

try:
    print("✓", subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip())
except FileNotFoundError:
    print("✗ git 이 설치되어 있지 않습니다 → https://git-scm.com")

✓ pandas     3.0.6
✓ plotly     7.1.0


✓ streamlit  1.66.1
✓ git version 2.56.0.windows.2


## 1. 프로젝트 폴더 구조

GitHub 에 올릴 것은 **대시보드에 필요한 파일만** 담은 `olist_dashboard` 폴더 하나입니다(원본 데이터 124MB 는 올리지 않습니다).
이 노트북을 실행하면 아래 폴더와 파일이 자동으로 만들어집니다.

```
olist_dashboard/                ← GitHub 저장소(repository)가 될 폴더
├── app.py                      ← 대시보드 프로그램 (Streamlit)
├── requirements.txt            ← 배포 서버가 설치할 패키지 목록
├── README.md                   ← GitHub 저장소 첫 화면 설명
├── .gitignore                  ← git 이 무시할 파일 목록
├── .streamlit/config.toml      ← 대시보드 색상 테마
└── data/
    ├── orders.csv.gz           ← 주문 단위 데이터 (가공 · 압축)
    └── items.csv.gz            ← 상품 단위 데이터 (가공 · 압축)
```

In [2]:
import os
import pandas as pd

PATH = "../00_data/"            # 원본 데이터 폴더
APP_DIR = "olist_dashboard"     # 만들 프로젝트 폴더

os.makedirs(f"{APP_DIR}/data", exist_ok=True)          # exist_ok=True: 이미 있어도 오류 없이 넘어감
os.makedirs(f"{APP_DIR}/.streamlit", exist_ok=True)
print("폴더 준비 완료:", os.path.abspath(APP_DIR))

폴더 준비 완료: C:\works\numpy\ecommerce\olist_분석_배포\olist_dashboard


---
## 2. 배포용 데이터 만들기

**왜 원본을 그대로 쓰지 않을까?**

- GitHub 는 **파일 하나가 100MB** 를 넘으면 올릴 수 없고, 50MB 가 넘으면 경고합니다(geolocation 파일 61MB).
- 무료 배포 서버는 메모리가 넉넉하지 않습니다. 접속할 때마다 큰 CSV 를 읽으면 느려집니다.
- 대시보드에 필요한 것은 **분석이 끝난 몇 개의 열**뿐입니다.

그래서 01 · 02 노트북에서 한 전처리를 한 번에 실행해 **필요한 열만 남긴 작은 표 2개**를 만듭니다.

| 파일 | 한 행 | 열 |
|---|---|---|
| `orders.csv.gz` | 배송 완료 주문 1건 | 주문 번호 · 주문 시각 · 고객 지역 · 결제 수단 · 주문 금액 · 배송 기간 · 지연 여부 · 리뷰 점수 |
| `items.csv.gz` | 팔린 상품 1개 | 주문 번호 · 카테고리 · 가격 · 배송비 |

In [3]:
# ① 원본 불러오기 — parse_dates=[...]: 읽으면서 바로 날짜형으로 변환
date_cols = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(PATH + "olist_orders_dataset.csv", parse_dates=date_cols)
customers = pd.read_csv(PATH + "olist_customers_dataset.csv")
payments = pd.read_csv(PATH + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(PATH + "olist_order_reviews_dataset.csv")
items = pd.read_csv(PATH + "olist_order_items_dataset.csv")
products = pd.read_csv(PATH + "olist_products_dataset.csv")
category = pd.read_csv(PATH + "product_category_name_translation.csv")
print("원본 주문 수:", len(orders))

원본 주문 수: 99441


In [4]:
# ② 분석 대상: 배송 완료 + 배송 완료일 기록 있음 + 데이터가 온전한 2017.01 ~ 2018.08
ts = orders["order_purchase_timestamp"]
dash = orders[(orders["order_status"] == "delivered")
              & orders["order_delivered_customer_date"].notna()
              & (ts >= "2017-01-01") & (ts < "2018-09-01")].copy()

# ③ 주문별 값 만들기 (주문 번호 → 값 대응표 = Series)
order_value = payments.groupby("order_id")["payment_value"].sum()                         # 주문 금액
main_type = (payments[payments["payment_type"] != "not_defined"]
             .sort_values("payment_value", ascending=False)
             .drop_duplicates("order_id")
             .set_index("order_id")["payment_type"])                                       # 주 결제 수단
review = reviews.groupby("order_id")["review_score"].mean()                                # 리뷰 점수

# ④ map(): 주문 번호를 대응표에서 찾아 값을 채워 넣음 (merge 보다 간단한 1:1 붙이기)
dash["payment_value"] = dash["order_id"].map(order_value)
dash["payment_type"] = dash["order_id"].map(main_type)
dash["review_score"] = dash["order_id"].map(review)
dash = dash.merge(customers[["customer_id", "customer_state"]], on="customer_id")

# ⑤ 배송 기간(일)과 지연 여부
dash["delivery_days"] = ((dash["order_delivered_customer_date"] - dash["order_purchase_timestamp"])
                         .dt.total_seconds() / 86400).round(2)
dash["late"] = dash["order_delivered_customer_date"] > dash["order_estimated_delivery_date"]

dash = dash.dropna(subset=["payment_value"])          # 결제 기록이 없는 주문 1건 제외
print("대시보드용 주문 수:", len(dash))

대시보드용 주문 수: 96203


**파일 크기 줄이기 요령** — `order_id` 는 `"e481f51cbdc54678b7cc49136f2d6af7"` 처럼 32글자라 파일에서 가장 큰 부분입니다.
대시보드에서는 주문을 구분만 하면 되므로 **1, 2, 3 … 짧은 번호**로 바꿉니다. items 표에도 **같은 번호**를 써야 두 표를 연결할 수 있습니다.

In [5]:
# 긴 주문 번호 → 짧은 정수 번호 대응표
new_id = pd.Series(range(1, len(dash) + 1), index=dash["order_id"])

orders_out = pd.DataFrame({
    "order_id": dash["order_id"].map(new_id),
    "purchase_ts": dash["order_purchase_timestamp"],
    "customer_state": dash["customer_state"],
    "payment_type": dash["payment_type"],
    "payment_value": dash["payment_value"].round(2),
    "delivery_days": dash["delivery_days"],
    "late": dash["late"],
    "review_score": dash["review_score"].round(2),
})

# ⑥ 상품 단위 표: 대시보드 주문에 들어 있는 상품만 + 카테고리(영어, 없으면 포르투갈어/unknown)
prod = products.merge(category, on="product_category_name", how="left")
prod["category"] = prod["product_category_name_english"].fillna(prod["product_category_name"]).fillna("unknown")
items_out = (items[items["order_id"].isin(new_id.index)]
             .merge(prod[["product_id", "category"]], on="product_id", how="left"))
items_out = pd.DataFrame({
    "order_id": items_out["order_id"].map(new_id),
    "category": items_out["category"],
    "price": items_out["price"],
    "freight_value": items_out["freight_value"],
})

# ⑦ 저장 — 파일 이름이 .gz 로 끝나면 pandas 가 자동으로 압축해서 저장
orders_out.to_csv(f"{APP_DIR}/data/orders.csv.gz", index=False)
items_out.to_csv(f"{APP_DIR}/data/items.csv.gz", index=False)

for f in ["orders.csv.gz", "items.csv.gz"]:
    size = os.path.getsize(f"{APP_DIR}/data/{f}") / 1024 / 1024
    print(f"{f:15s} {size:5.2f} MB")
orders_out.head()

orders.csv.gz    1.51 MB
items.csv.gz     0.89 MB


,order_id,purchase_ts,customer_state,payment_type,payment_value,delivery_days,late,review_score
0,1,2017-10-02 10:56:33,SP,voucher,38.71,8.44,False,4.0
1,2,2018-07-24 20:41:37,BA,boleto,141.46,13.78,False,4.0
2,3,2018-08-08 08:38:49,GO,credit_card,179.12,9.39,False,5.0
3,4,2017-11-18 19:28:06,RN,credit_card,72.20,13.21,False,5.0
4,5,2018-02-13 21:18:39,SP,credit_card,28.62,2.87,False,5.0


<div style="border:1px solid #F0D58C;border-left:5px solid #E0A800;background:#FFF8E1;padding:10px 14px;border-radius:4px">

**데이터를 공개 저장소에 올릴 때 확인할 것** — ① 개인 정보가 없는지(이 데이터는 이미 익명화됨) ② 데이터 **라이선스**가 공개를 허용하는지.
Olist 데이터는 CC BY-NC-SA 4.0 (비영리 · 출처 표시 · 같은 조건으로 공유) 이므로 대시보드 하단과 README 에 **출처를 표시**합니다. 회사 데이터라면 반드시 허가를 받으세요.

</div>

---
## 3. 대시보드 프로그램 `app.py` 만들기

**Streamlit** 은 파이썬 코드만으로 웹 화면을 만드는 라이브러리입니다. 핵심 원리는 하나입니다.

> 사용자가 필터 · 버튼 등을 조작할 때마다 **app.py 를 위에서부터 다시 실행**해 화면을 새로 그린다.

| 사용한 기능 | 역할 |
|---|---|
| `st.set_page_config` | 브라우저 탭 제목 · 아이콘 · 넓은 화면 |
| `@st.cache_data` | 데이터 읽기 결과를 기억(캐시) → 다시 실행돼도 빠름 |
| `st.sidebar.date_input` · `multiselect` | 왼쪽 사이드바의 기간 · 지역 필터 |
| `st.columns` + `st.metric` | 핵심 지표(KPI) 6개를 가로로 배치 |
| `st.tabs` | 매출 · 시간 · 카테고리 · 배송 · 지역 · 데이터 탭 |
| `st.radio` · `slider` · `selectbox` | 탭 안에서 보는 기준 바꾸기 |
| `st.plotly_chart` | 02 노트북에서 만든 plotly 그래프를 그대로 표시 |
| `st.download_button` | 필터 결과를 CSV 로 내려받기 |

아래 셀의 첫 줄 `%%writefile 경로` 는 **셀 내용을 파일로 저장**하는 주피터 명령입니다(셀을 실행해도 코드가 실행되지는 않고 파일만 만들어짐).

In [6]:
%%writefile olist_dashboard/app.py
"""
Olist 브라질 이커머스 대시보드 (Streamlit)

실행 방법 (터미널):  streamlit run app.py
필요한 데이터:       data/orders.csv.gz, data/items.csv.gz  (03 노트북에서 생성)
"""
from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st

# ── 1. 페이지 기본 설정 (반드시 다른 st 명령보다 먼저) ─────────────────────
st.set_page_config(page_title="Olist 이커머스 대시보드", page_icon="📦", layout="wide")

# app.py 가 있는 폴더 기준으로 data 폴더를 찾음 → 어느 위치에서 실행해도, 배포 서버에서도 경로가 맞음
DATA_DIR = Path(__file__).parent / "data"
WEEKDAYS = ["월", "화", "수", "목", "금", "토", "일"]


# ── 2. 데이터 불러오기 ────────────────────────────────────────────────
@st.cache_data  # 한 번 읽은 결과를 기억해 두고 재사용 → 필터를 바꿀 때마다 파일을 다시 읽지 않아 빠름
def load_data():
    orders = pd.read_csv(DATA_DIR / "orders.csv.gz", parse_dates=["purchase_ts"])
    items = pd.read_csv(DATA_DIR / "items.csv.gz")
    return orders, items


orders, items = load_data()

# ── 3. 사이드바: 필터 ────────────────────────────────────────────────
st.sidebar.title("🔎 필터")
first_day = orders["purchase_ts"].min().date()
last_day = orders["purchase_ts"].max().date()
period = st.sidebar.date_input("주문 기간", value=(first_day, last_day),
                               min_value=first_day, max_value=last_day)
if len(period) != 2:                       # 달력에서 시작일만 고른 순간에는 날짜가 1개
    st.info("👈 주문 기간의 끝 날짜까지 선택해 주세요.")
    st.stop()                              # 여기서 화면 그리기를 멈춤
start, end = period

all_states = sorted(orders["customer_state"].unique())
picked_states = st.sidebar.multiselect("고객 지역 (주)", all_states, placeholder="전체 지역")
st.sidebar.caption("지역을 고르지 않으면 전체 지역을 보여 줍니다.")

# 필터 적용: 조건에 맞는 행만 True
mask = orders["purchase_ts"].dt.date.between(start, end)
if picked_states:
    mask &= orders["customer_state"].isin(picked_states)
f_orders = orders[mask]
f_items = items[items["order_id"].isin(f_orders["order_id"])]

if f_orders.empty:
    st.warning("조건에 맞는 주문이 없습니다. 필터를 바꿔 보세요.")
    st.stop()

# ── 4. 제목과 핵심 지표(KPI) ─────────────────────────────────────────
st.title("📦 Olist 브라질 이커머스 대시보드")
region = "전체 지역" if not picked_states else ", ".join(picked_states)
st.caption(f"배송 완료 주문 기준 · {start} ~ {end} · {region}")

k1, k2, k3 = st.columns(3)          # 지표 6개를 3개씩 두 줄로 (한 줄에 6개면 숫자가 잘림)
k4, k5, k6 = st.columns(3)
k1.metric("주문 수", f"{len(f_orders):,}건")
k2.metric("매출", f"R$ {f_orders['payment_value'].sum() / 1e6:,.2f}M")
k3.metric("객단가 (평균)", f"R$ {f_orders['payment_value'].mean():,.0f}")
k4.metric("배송 기간 (중앙값)", f"{f_orders['delivery_days'].median():.1f}일")
k5.metric("배송 지연율", f"{f_orders['late'].mean() * 100:.1f}%")
k6.metric("평균 리뷰", f"{f_orders['review_score'].mean():.2f}점")

# ── 5. 탭별 분석 화면 ───────────────────────────────────────────────
tab_sales, tab_time, tab_cat, tab_ship, tab_state, tab_data = st.tabs(
    ["📈 매출 추이", "🕒 주문 시간", "🛍️ 카테고리", "🚚 배송·리뷰", "🗺️ 지역", "📄 데이터"])

# (1) 매출 추이
with tab_sales:
    choice = st.radio("보고 싶은 지표", ["매출", "주문 수", "객단가"], horizontal=True)
    # "MS"(Month Start): 각 달을 1일 날짜로 표시 → x축 '2017-01' 눈금과 막대 위치가 정확히 맞음
    monthly = (f_orders.set_index("purchase_ts").resample("MS")
               .agg(revenue=("payment_value", "sum"), orders=("order_id", "count"),
                    aov=("payment_value", "mean"))
               .reset_index())
    column = {"매출": "revenue", "주문 수": "orders", "객단가": "aov"}[choice]
    fig = px.bar(monthly, x="purchase_ts", y=column, labels={"purchase_ts": "주문 월", column: choice},
                 title=f"월별 {choice}")
    fig.update_xaxes(dtick="M1", tickformat="%Y-%m", tickangle=-45)
    st.plotly_chart(fig, width="stretch")

# (2) 주문 시간 — 요일 × 시간 히트맵
with tab_time:
    t = f_orders.assign(weekday=f_orders["purchase_ts"].dt.dayofweek.map(dict(enumerate(WEEKDAYS))),
                        hour=f_orders["purchase_ts"].dt.hour)
    heat = (t.pivot_table(index="weekday", columns="hour", values="order_id", aggfunc="count")
            .reindex(index=WEEKDAYS, columns=range(24)).fillna(0))
    fig = px.imshow(heat, labels=dict(x="시", y="요일", color="주문 수"), aspect="auto",
                    color_continuous_scale="Blues", title="요일 × 시간대 주문 수")
    fig.update_xaxes(dtick=1)
    st.plotly_chart(fig, width="stretch")
    st.caption("진한 칸일수록 주문이 많은 시간대입니다. 프로모션 · 고객센터 운영 시간 결정에 활용할 수 있습니다.")

# (3) 카테고리
with tab_cat:
    left, right = st.columns([1, 3])
    basis = left.radio("순위 기준", ["매출", "판매 수량"])
    top_n = left.slider("표시할 개수", min_value=5, max_value=30, value=15)
    cat = f_items.groupby("category").agg(revenue=("price", "sum"), items=("order_id", "size"))
    col = "revenue" if basis == "매출" else "items"
    top = cat.sort_values(col, ascending=False).head(top_n).iloc[::-1]   # 가로 막대: 1위가 맨 위에 오도록 뒤집기
    fig = px.bar(top, x=col, y=top.index, orientation="h",
                 labels={col: basis, "category": "카테고리"}, title=f"카테고리 {basis} TOP {top_n}")
    fig.update_layout(height=max(400, top_n * 24))
    right.plotly_chart(fig, width="stretch")

# (4) 배송과 리뷰
with tab_ship:
    left, right = st.columns(2)
    bins = [0, 7, 14, 21, 30, 60, 1000]
    labels = ["~7일", "8~14일", "15~21일", "22~30일", "31~60일", "60일+"]
    groups = pd.cut(f_orders["delivery_days"], bins=bins, labels=labels)
    by_group = f_orders.groupby(groups, observed=True)["review_score"].mean().reset_index()
    fig = px.bar(by_group, x="delivery_days", y="review_score", text_auto=".2f",
                 labels={"delivery_days": "배송 기간", "review_score": "평균 리뷰 점수"},
                 title="배송 기간별 평균 리뷰 점수")
    fig.update_yaxes(range=[1, 5])
    left.plotly_chart(fig, width="stretch")

    late_rate = (f_orders.set_index("purchase_ts").resample("MS")["late"].mean() * 100).reset_index()
    fig = px.line(late_rate, x="purchase_ts", y="late", markers=True,
                  labels={"purchase_ts": "주문 월", "late": "지연율 (%)"}, title="월별 배송 지연율")
    right.plotly_chart(fig, width="stretch")

    on_time = f_orders.loc[~f_orders["late"], "review_score"].mean()
    delayed = f_orders.loc[f_orders["late"], "review_score"].mean()
    st.info(f"제때 도착한 주문의 평균 리뷰는 **{on_time:.2f}점**, 지연된 주문은 **{delayed:.2f}점**입니다.")

# (5) 지역
with tab_state:
    state = f_orders.groupby("customer_state").agg(
        orders=("order_id", "count"), revenue=("payment_value", "sum"),
        median_days=("delivery_days", "median"), late_rate=("late", "mean"),
        review=("review_score", "mean"))
    state["late_rate"] *= 100
    names = {"주문 수": "orders", "매출": "revenue", "배송 기간 (중앙값)": "median_days",
             "지연율 (%)": "late_rate", "평균 리뷰": "review"}
    pick = st.selectbox("비교할 지표", list(names))
    s = state.sort_values(names[pick], ascending=False)
    fig = px.bar(s, x=s.index, y=names[pick], labels={"x": "주(state)", names[pick]: pick},
                 title=f"주(state)별 {pick}")
    st.plotly_chart(fig, width="stretch")
    st.dataframe(state.rename(columns={v: k for k, v in names.items()}).round(2), width="stretch")

# (6) 원본 데이터 확인 · 내려받기
with tab_data:
    st.write(f"필터가 적용된 주문 **{len(f_orders):,}건** 중 앞의 1,000건")
    st.dataframe(f_orders.head(1000), width="stretch")
    st.download_button("⬇️ 필터 결과 CSV 내려받기",
                       data=f_orders.to_csv(index=False).encode("utf-8-sig"),   # utf-8-sig: 엑셀에서 한글이 안 깨짐
                       file_name="olist_filtered_orders.csv", mime="text/csv")

st.divider()
st.caption("데이터: Olist Brazilian E-Commerce Public Dataset (Kaggle, CC BY-NC-SA 4.0) · 교육용 분석 대시보드")

Writing olist_dashboard/app.py


## 4. 배포 설정 파일 만들기

### requirements.txt — 배포 서버가 설치할 패키지 목록
배포 서버(Streamlit Cloud)는 빈 컴퓨터입니다. 이 파일을 보고 필요한 패키지를 설치합니다. **빠뜨리면 `ModuleNotFoundError`** 로 앱이 실행되지 않습니다.

In [7]:
%%writefile olist_dashboard/requirements.txt
streamlit>=1.50
pandas>=2.2
plotly>=5.24

Writing olist_dashboard/requirements.txt


### .gitignore — git 이 무시할 파일
실행할 때 자동으로 생기는 임시 파일, 가상환경 폴더, **비밀번호 · API 키 파일**(`secrets.toml`)은 GitHub 에 올리면 안 됩니다.

In [8]:
%%writefile olist_dashboard/.gitignore
# 파이썬 실행 중 생기는 임시 파일
__pycache__/
*.pyc
.ipynb_checkpoints/

# 가상환경 폴더
.venv/
venv/

# 운영체제가 만드는 파일
.DS_Store
Thumbs.db

# 비밀 정보 (API 키 등) — 절대 올리지 않기
.streamlit/secrets.toml

Writing olist_dashboard/.gitignore


### .streamlit/config.toml — 화면 테마 (선택)

In [9]:
%%writefile olist_dashboard/.streamlit/config.toml
[theme]
base = "light"
primaryColor = "#F24472"

[server]
headless = true

Writing olist_dashboard/.streamlit/config.toml


### README.md — 저장소 첫 화면에 보이는 설명서
GitHub 저장소에 들어오면 가장 먼저 보이는 문서입니다. 무엇을 · 어떻게 실행하는지 · 데이터 출처를 적습니다. 배포 후 받은 주소를 `배포 주소` 줄에 적어 두세요.

In [10]:
%%writefile olist_dashboard/README.md
# 📦 Olist 브라질 이커머스 대시보드

브라질 온라인 마켓플레이스 Olist 의 주문 데이터(2017.01 ~ 2018.08, 배송 완료 주문 약 9.6만 건)를 분석한 Streamlit 대시보드입니다.

- **배포 주소**: (배포 후 여기에 https://....streamlit.app 주소를 적으세요)

## 주요 기능
- 기간 · 지역(주) 필터
- 핵심 지표: 주문 수, 매출, 객단가, 배송 기간, 지연율, 평균 리뷰
- 탭: 월별 매출 추이 / 요일 × 시간 히트맵 / 카테고리 순위 / 배송 기간과 리뷰 / 지역 비교 / 데이터 내려받기

## 로컬에서 실행하기
```bash
pip install -r requirements.txt
streamlit run app.py
```

## 주요 인사이트
- 배송이 지연되면 평균 리뷰가 약 4.3점 → 2.6점으로 하락
- 재구매율 약 3% — 재구매 유도가 핵심 과제
- 2017년 11월 블랙프라이데이에 주문 · 배송 지연 급증

## 데이터 출처
Olist, *Brazilian E-Commerce Public Dataset by Olist* (Kaggle), CC BY-NC-SA 4.0.
대시보드용으로 필요한 열만 가공 · 압축해 `data/` 에 담았습니다.

Writing olist_dashboard/README.md


---
## 5. 앱 테스트하기

### 5-1. 자동 테스트 — `AppTest`
브라우저를 열지 않고 앱을 한 번 실행해 **오류가 없는지**, 필터를 바꿔도 잘 동작하는지 코드로 확인합니다. 배포 전에 꼭 해 볼 습관입니다.

In [11]:
from pathlib import Path
import streamlit.logger
from streamlit.testing.v1 import AppTest

streamlit.logger.set_log_level("error")     # 테스트 중 나오는 무해한 경고 메시지(ScriptRunContext …) 숨기기

app_path = str(Path(APP_DIR, "app.py").resolve())       # 절대 경로로 지정
at = AppTest.from_file(app_path, default_timeout=60)
at.run()                                                 # 앱을 위에서부터 한 번 실행

print("오류:", at.exception if at.exception else "없음 ✓")
for m in at.metric:                                      # 화면의 KPI 카드 읽기
    print(f"  {m.label:12s} {m.value}")

오류: 없음 ✓
  주문 수         96,203건
  매출           R$ 15.37M
  객단가 (평균)     R$ 160
  배송 기간 (중앙값)  10.2일
  배송 지연율       8.1%
  평균 리뷰        4.14점


In [12]:
# 사이드바 지역 필터에서 'SP'(상파울루)만 고른 것처럼 조작한 뒤 다시 실행
at.sidebar.multiselect[0].set_value(["SP"]).run()
print("오류:", at.exception if at.exception else "없음 ✓")
print("SP 필터 후:", {m.label: m.value for m in at.metric})

오류: 없음 ✓
SP 필터 후: {'주문 수': '40,399건', '매출': 'R$ 5.76M', '객단가 (평균)': 'R$ 142', '배송 기간 (중앙값)': '7.2일', '배송 지연율': '5.9%', '평균 리뷰': '4.24점'}


### 5-2. 내 컴퓨터에서 직접 띄워 보기

`streamlit run` 은 서버를 계속 켜 두는 명령이라 노트북 셀이 아닌 **터미널**(VS Code 메뉴 → 터미널 → 새 터미널)에서 실행합니다.

```bash
cd olist_dashboard          # 이 노트북이 있는 폴더 안의 olist_dashboard 로 이동
streamlit run app.py
```

브라우저에 `http://localhost:8501` 이 열리면 성공입니다. 필터를 바꿔 보고, 끝내려면 터미널에서 **Ctrl + C** 를 누릅니다.
`app.py` 를 고치고 저장하면 브라우저 오른쪽 위에 **Rerun** 이 나타나 바로 반영할 수 있습니다.

---
## 6. git 과 GitHub 기본 개념

| 용어 | 뜻 | 비유 |
|---|---|---|
| **git** | 파일의 변경 이력을 기록하는 프로그램 (내 컴퓨터) | 게임의 '저장 슬롯' |
| **저장소 (repository)** | git 이 관리하는 프로젝트 폴더 | 저장 슬롯이 있는 게임 폴더 |
| **commit** | 현재 상태를 메시지와 함께 기록 | 저장하기 |
| **GitHub** | 저장소를 인터넷에 올려 두는 서비스 | 클라우드 저장 |
| **remote (origin)** | 내 저장소와 연결된 GitHub 저장소 주소 | 클라우드 계정 연결 |
| **push** | 내 commit 들을 GitHub 로 올리기 | 클라우드에 업로드 |
| **branch (main)** | 작업 줄기. 배포는 보통 `main` 브랜치 기준 | |

```
 파일 수정 ──git add──▶ 올릴 목록(stage) ──git commit──▶ 내 컴퓨터에 기록 ──git push──▶ GitHub
```

## 7. GitHub 에 올리기

### 7-1. (처음 한 번만) git 사용자 정보 등록
commit 에 '누가 저장했는지' 기록할 이름과 이메일입니다. **GitHub 가입 이메일**을 쓰세요. 터미널에서:

```bash
git config --global user.name "내 이름"
git config --global user.email "github가입이메일@example.com"
```

### 7-2. 내 컴퓨터에 저장소 만들고 첫 commit
**반드시 `olist_dashboard` 폴더 안에서** 실행합니다(상위 폴더에서 하면 원본 데이터까지 올라갑니다).

```bash
cd olist_dashboard
git init                                   # 이 폴더를 git 저장소로 만들기 (.git 폴더 생성)
git add .                                  # 모든 파일을 올릴 목록에 추가 (.gitignore 에 적힌 것은 제외)
git status                                 # 무엇이 올라갈지 확인 — app.py, data/, requirements.txt … 가 보여야 함
git commit -m "Olist 대시보드 첫 버전"       # 메시지와 함께 저장
```

### 7-3. GitHub 에 빈 저장소 만들기
1. https://github.com 로그인 → 오른쪽 위 **+** → **New repository**
2. Repository name: `olist-dashboard` (영문 · 숫자 · 하이픈 권장)
3. **Public** 선택 (Streamlit 무료 배포는 공개 저장소가 가장 간단)
4. *Add a README file* 등은 **체크하지 않음** (이미 README.md 가 있음 — 체크하면 첫 push 가 거절될 수 있음)
5. **Create repository** → 화면에 나오는 주소 `https://github.com/내아이디/olist-dashboard.git` 복사

### 7-4. 연결하고 올리기 (push)
```bash
git branch -M main                                                    # 브랜치 이름을 main 으로
git remote add origin https://github.com/내아이디/olist-dashboard.git  # GitHub 저장소 연결 (처음 한 번)
git push -u origin main                                               # 올리기 (-u: 다음부터는 git push 만 쳐도 됨)
```
처음 push 하면 GitHub 로그인 창이 뜹니다. 브라우저로 로그인하면 됩니다.
GitHub 저장소 페이지를 새로 고쳐 파일들과 README 가 보이면 성공입니다.

<div style="border:1px solid #F0D58C;border-left:5px solid #E0A800;background:#FFF8E1;padding:10px 14px;border-radius:4px">

**올리기 전 체크** — `git status` 에 `secrets.toml`, 가상환경 폴더(`venv/`), 큰 원본 CSV 가 보이면 안 됩니다. 한 번 GitHub 에 올라간 비밀 정보는 지워도 기록에 남습니다.

</div>

---
## 8. Streamlit Community Cloud 로 배포하기

1. https://share.streamlit.io 접속 → **Continue with GitHub** 로 로그인 (처음이면 GitHub 접근 권한 허용)
2. 오른쪽 위 **Create app** → **Deploy a public app from GitHub** (또는 *Yup, I have an app*)
3. 입력
   - **Repository**: `내아이디/olist-dashboard`
   - **Branch**: `main`
   - **Main file path**: `app.py`
   - **App URL**: 원하는 주소 (예: `olist-dashboard-내이름`) → `https://olist-dashboard-내이름.streamlit.app`
4. **Advanced settings** → Python version 을 **3.12** 로 선택 (내 컴퓨터와 같게)
5. **Deploy** 클릭 → 1~3분 동안 서버가 `requirements.txt` 의 패키지를 설치하고 앱을 실행
6. 대시보드가 뜨면 주소를 복사해 친구 · 동료에게 공유하고, README 의 `배포 주소` 줄에도 적어 두세요.

배포 중 오른쪽 아래 **Manage app** 을 누르면 서버 로그(설치 과정 · 오류 메시지)를 볼 수 있습니다.

## 9. 수정하고 다시 배포하기 — git push 만 하면 끝

배포된 앱은 GitHub 의 `main` 브랜치를 지켜보고 있다가, **새 commit 이 push 되면 자동으로 다시 배포**합니다.

```bash
# 예: app.py 의 제목을 고친 뒤
git add app.py
git commit -m "대시보드 제목 수정"
git push
```
1분쯤 뒤 배포 주소를 새로 고치면 바뀐 화면이 보입니다. 데이터를 새로 만들었다면 `data/` 폴더도 함께 add · commit 합니다.

## 10. 자주 만나는 문제와 해결

| 증상 | 원인 | 해결 |
|---|---|---|
| 배포 로그에 `ModuleNotFoundError: No module named 'plotly'` | requirements.txt 에 패키지가 빠짐 | requirements.txt 에 추가 → commit → push |
| `FileNotFoundError: data/orders.csv.gz` | ① data 폴더를 push 하지 않음 ② 실행 위치 기준 상대 경로 사용 | ① `git add data` 후 push ② `Path(__file__).parent / "data"` 처럼 **app.py 기준 경로** 사용 (이 앱은 이미 적용) |
| `git push` 가 `rejected (fetch first)` 로 거절 | GitHub 저장소에 내 컴퓨터에 없는 commit 이 있음 (예: README 를 체크하고 만듦) | `git pull --rebase origin main` 후 다시 `git push` |
| `remote origin already exists` | remote 를 두 번 추가 | `git remote set-url origin 새주소` |
| `fatal: not a git repository` | 저장소 폴더 밖에서 명령 실행 | `cd olist_dashboard` 로 이동 후 실행 |
| `warning: LF will be replaced by CRLF` | Windows 줄바꿈 차이 알림 | 무시해도 됨 (오류 아님) |
| `File ... exceeds GitHub's file size limit of 100.00 MB` | 큰 파일을 commit 함 | 데이터를 줄이거나 압축, `.gitignore` 에 추가 |
| 며칠 뒤 접속하니 "This app has gone to sleep" | 무료 배포는 오래 접속이 없으면 잠듦 | 버튼을 눌러 깨우면 1분 내 다시 실행 |
| 화면이 매번 느림 | 데이터 읽기를 매번 다시 함 | `@st.cache_data` 사용 (이 앱은 이미 적용) |

<div style="border:1px solid #A5D6A7;border-left:5px solid #2E7D32;background:#F1F8F2;padding:10px 14px;border-radius:4px">

**전체 과정 요약**

1. 노트북으로 분석 → **2.** 배포용 작은 데이터 저장 → **3.** `app.py` · `requirements.txt` · `.gitignore` 작성
4. `AppTest` 와 `streamlit run` 으로 테스트 → **5.** `git init / add / commit` → **6.** GitHub 저장소 만들고 `push`
7. Streamlit Cloud 에서 저장소 · 브랜치 · `app.py` 를 골라 **Deploy** → **8.** 고칠 때는 `commit` + `push` 만 하면 자동 재배포

</div>

# 도전 과제

1. **결제 수단 필터 추가** — 사이드바에 `payment_type` multiselect 를 추가해 보세요(지역 필터 코드를 참고).
2. **재구매율 지표** — 01 노트북의 재구매율을 KPI 로 추가해 보세요. (힌트: 배포용 데이터에 `customer_unique_id` 를 짧은 번호로 바꿔 함께 저장해야 합니다.)
3. **카테고리 탭 개선** — 카테고리별 **평균 리뷰 점수**와 **지연율**을 표로 함께 보여 주세요.
4. 고친 앱을 `commit` · `push` 해서 배포된 주소에 반영되는지 확인하세요.